In [2]:
import pandas as pd
from pathlib import Path

In [3]:
parent_path = Path.cwd().parent

In [4]:
dataset = pd.read_csv(parent_path / 'data' / 'website_conversions.csv')
ppc = pd.read_csv(parent_path / 'data' / 'ppc_spend.csv')
ppc['channel'] = 'PPC'
ppc.rename(columns={'spend': 'ppc_spend'}, inplace=True)
email = pd.read_csv(parent_path / 'data' /  'email_campaigns.csv')
email['channel'] = 'Email'
email.rename(columns={'clicks': 'email_clicks'}, inplace=True)
social_media = pd.read_csv(parent_path / 'data' / 'social_media_ads.csv')
social_media['channel'] = 'Social Media'
social_media.rename(columns={
    'clicks': 'social_media_clicks', 'spend': 'social_media_spend'
    }, inplace=True)

In [5]:
dataset.head()

,date,conversion_id,channel,revenue
0,2024-08-01,CONV-1,PPC,73.80
1,2024-08-01,CONV-2,PPC,69.11
2,2024-08-01,CONV-3,PPC,75.05
3,2024-08-01,CONV-4,PPC,79.78
4,2024-08-01,CONV-5,PPC,58.87


In [6]:
dataset = dataset.groupby(['date', 'channel']).agg(
    conversions=('conversion_id', 'count'),
    revenue=('revenue', 'sum')
)
dataset.head()

conversions  revenue
date       channel                           
2024-08-01 Email                   9   978.96
           PPC                    11  1045.63
           Social Media            4   268.02
2024-08-02 Email                   8   879.14
           PPC                    24  2124.40

In [7]:
dataset = pd.merge(dataset, ppc, on=['date', 'channel'], how='left')
dataset = pd.merge(dataset, email, on=['date', 'channel'], how='left')
dataset = pd.merge(dataset, social_media, on=['date', 'channel'], how='left')
dataset.head()

,date,channel,conversions,revenue,ppc_spend,email_clicks,emails_sent,social_media_spend,social_media_clicks,impressions
0,2024-08-01,Email,9,978.96,NaN,110.000000,1999.0,NaN,NaN,NaN
1,2024-08-01,PPC,11,1045.63,250.82,NaN,NaN,NaN,NaN,NaN
2,2024-08-01,Social Media,4,268.02,NaN,NaN,NaN,172.82,228.0,16755.0
3,2024-08-02,Email,8,879.14,NaN,90.999583,1845.0,NaN,NaN,NaN
4,2024-08-02,PPC,24,2124.40,366.19,NaN,NaN,NaN,NaN,NaN


In [8]:
dataset['spend'] = dataset['ppc_spend'].fillna(0) + dataset['social_media_spend'].fillna(0)
dataset['clicks'] = dataset['email_clicks'].fillna(0) + dataset['social_media_clicks'].fillna(0)
dataset.drop(columns=['ppc_spend', 'social_media_spend', 'social_media_clicks', 'email_clicks'], inplace=True)
dataset.head()

,date,channel,conversions,revenue,emails_sent,impressions,spend,clicks
0,2024-08-01,Email,9,978.96,1999.0,NaN,0.00,110.000000
1,2024-08-01,PPC,11,1045.63,NaN,NaN,250.82,0.000000
2,2024-08-01,Social Media,4,268.02,NaN,16755.0,172.82,228.000000
3,2024-08-02,Email,8,879.14,1845.0,NaN,0.00,90.999583
4,2024-08-02,PPC,24,2124.40,NaN,NaN,366.19,0.000000


In [10]:
# create week column based on date
dataset['date'] = pd.to_datetime(dataset['date'])
dataset['week'] = dataset['date'].dt.strftime('%G-%V')
dataset.head()

,date,channel,conversions,revenue,emails_sent,impressions,spend,clicks,week
0,2024-08-01,Email,9,978.96,1999.0,NaN,0.00,110.000000,2024-31
1,2024-08-01,PPC,11,1045.63,NaN,NaN,250.82,0.000000,2024-31
2,2024-08-01,Social Media,4,268.02,NaN,16755.0,172.82,228.000000,2024-31
3,2024-08-02,Email,8,879.14,1845.0,NaN,0.00,90.999583,2024-31
4,2024-08-02,PPC,24,2124.40,NaN,NaN,366.19,0.000000,2024-31


In [11]:
#export the dataset to .csv
dataset.to_csv(parent_path / 'data' / 'aggregate_data.csv', index=False)